<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Finance (2nd ed.)
**Mastering Data-Driven Finance**
&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH
<img src="http://hilpisch.com/images/py4fi_2nd_shadow.png" width="300px" align="left">

# DX Analytics

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt

In [ ]:
from pylab import mpl, plt
plt.style.use('seaborn-v0_8')
mpl.rcParams['font.family'] = 'serif'
%config InlineBackend.figure_format = 'svg'

In [ ]:
import sys
sys.path.append('../dx')

## DX Frame

### Risk-Neutral Discounting

In [ ]:
dates = [dt.datetime(2020, 1, 1), dt.datetime(2020, 7, 1),
         dt.datetime(2021, 1, 1)]

In [ ]:
(dates[1] - dates[0]).days / 365.

In [ ]:
(dates[2] - dates[1]).days / 365.

In [ ]:
fractions = [0.0, 0.5, 1.0]

In [ ]:
from get_year_deltas import get_year_deltas

In [ ]:
get_year_deltas(dates)

In [ ]:
from constant_short_rate import constant_short_rate

In [ ]:
csr = constant_short_rate('csr', 0.05)

In [ ]:
csr.get_discount_factors(dates)

In [ ]:
deltas = get_year_deltas(dates)
deltas

In [ ]:
csr.get_discount_factors(deltas, dtobjects=False)

### Market Environment

In [ ]:
from market_environment import market_environment

In [ ]:
me = market_environment('me_gbm', dt.datetime(2020, 1, 1))

In [ ]:
me.add_constant('initial_value', 36.)

In [ ]:
me.add_constant('volatility', 0.2)

In [ ]:
me.add_constant('final_date', dt.datetime(2020, 12, 31))

In [ ]:
me.add_constant('currency', 'EUR')

In [ ]:
me.add_constant('frequency', 'M')

In [ ]:
me.add_constant('paths', 10000)

In [ ]:
me.add_curve('discount_curve', csr)

In [ ]:
me.get_constant('volatility')

In [ ]:
me.get_curve('discount_curve').short_rate

## DX Simulation

In [ ]:
from sn_random_numbers import *

In [ ]:
snrn = sn_random_numbers((2, 2, 2), antithetic=False,
                         moment_matching=False, fixed_seed=True)
snrn

In [ ]:
round(snrn.mean(), 6)

In [ ]:
round(snrn.std(), 6)

In [ ]:
snrn = sn_random_numbers((2, 2, 2), antithetic=False,
                         moment_matching=True, fixed_seed=True)
snrn

In [ ]:
round(snrn.mean(), 6)

In [ ]:
round(snrn.std(), 6)

### Geometric Brownian Motion

In [ ]:
from dx_frame import *

In [ ]:
me_gbm = market_environment('me_gbm', dt.datetime(2020, 1, 1))

In [ ]:
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('volatility', 0.2)
me_gbm.add_constant('final_date', dt.datetime(2020, 12, 31))
me_gbm.add_constant('currency', 'EUR')
me_gbm.add_constant('frequency', 'M')
me_gbm.add_constant('paths', 10000)

In [ ]:
csr = constant_short_rate('csr', 0.06)

In [ ]:
me_gbm.add_curve('discount_curve', csr)

In [ ]:
from geometric_brownian_motion import geometric_brownian_motion

In [ ]:
gbm = geometric_brownian_motion('gbm', me_gbm)

In [ ]:
gbm.generate_time_grid()

In [ ]:
gbm.time_grid

In [ ]:
%time paths_1 = gbm.get_instrument_values()

In [ ]:
paths_1.round(3)

In [ ]:
gbm.update(volatility=0.5)

In [ ]:
%time paths_2 = gbm.get_instrument_values()

In [ ]:
plt.figure(figsize=(10, 6))
p1 = plt.plot(gbm.time_grid, paths_1[:, :10], 'b')
p2 = plt.plot(gbm.time_grid, paths_2[:, :10], 'r-.')
l1 = plt.legend([p1[0], p2[0]],
                ['low volatility', 'high volatility'], loc=2)
plt.gca().add_artist(l1)
plt.xticks(rotation=30);

### Jump Diffusion

In [ ]:
me_jd = market_environment('me_jd', dt.datetime(2020, 1, 1))

In [ ]:
# specific to simulation class
me_jd.add_constant('lambda', 0.3)
me_jd.add_constant('mu', -0.75)
me_jd.add_constant('delta', 0.1)

In [ ]:
me_jd.add_environment(me_gbm)

In [ ]:
from jump_diffusion import jump_diffusion

In [ ]:
jd = jump_diffusion('jd', me_jd)

In [ ]:
%time paths_3 = jd.get_instrument_values()

In [ ]:
jd.update(lamb=0.9)

In [ ]:
%time paths_4 = jd.get_instrument_values()

In [ ]:
plt.figure(figsize=(10, 6))
p1 = plt.plot(gbm.time_grid, paths_3[:, :10], 'b')
p2 = plt.plot(gbm.time_grid, paths_4[:, :10], 'r-.')
l1 = plt.legend([p1[0], p2[0]],
                ['low intensity', 'high intensity'], loc=3)
plt.gca().add_artist(l1)
plt.xticks(rotation=30);

### Square-Root Diffusion

In [ ]:
me_srd = market_environment('me_srd', dt.datetime(2020, 1, 1))

In [ ]:
me_srd.add_constant('initial_value', .25)
me_srd.add_constant('volatility', 0.05)
me_srd.add_constant('final_date', dt.datetime(2020, 12, 31))
me_srd.add_constant('currency', 'EUR')
me_srd.add_constant('frequency', 'W')
me_srd.add_constant('paths', 10000)

In [ ]:
# specific to simualation class
me_srd.add_constant('kappa', 4.0)
me_srd.add_constant('theta', 0.2)

In [ ]:
me_srd.add_curve('discount_curve', constant_short_rate('r', 0.0))

In [ ]:
from square_root_diffusion import square_root_diffusion

In [ ]:
srd = square_root_diffusion('srd', me_srd)

In [ ]:
srd_paths = srd.get_instrument_values()[:, :10]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(srd.time_grid, srd.get_instrument_values()[:, :10])
plt.axhline(me_srd.get_constant('theta'), color='r',
            ls='--', lw=2.0)
plt.xticks(rotation=30);

## Valuation Classes

### European Options

In [ ]:
me_gbm = market_environment('me_gbm', dt.datetime(2020, 1, 1))

In [ ]:
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('volatility', 0.2)
me_gbm.add_constant('final_date', dt.datetime(2020, 12, 31))
me_gbm.add_constant('currency', 'EUR')
me_gbm.add_constant('frequency', 'M')
me_gbm.add_constant('paths', 10000)

In [ ]:
csr = constant_short_rate('csr', 0.06)

In [ ]:
me_gbm.add_curve('discount_curve', csr)

In [ ]:
gbm = geometric_brownian_motion('gbm', me_gbm)

In [ ]:
me_call = market_environment('me_call', me_gbm.pricing_date)

In [ ]:
me_call.add_constant('strike', 40.)
me_call.add_constant('maturity', dt.datetime(2020, 12, 31))
me_call.add_constant('currency', 'EUR')

In [ ]:
payoff_func = 'np.maximum(maturity_value - strike, 0)'

In [ ]:
from valuation_mcs_european import valuation_mcs_european

In [ ]:
eur_call = valuation_mcs_european('eur_call', underlying=gbm,
                        mar_env=me_call, payoff_func=payoff_func)

In [ ]:
%time eur_call.present_value()

In [ ]:
%time eur_call.delta()

In [ ]:
%time eur_call.vega()

In [ ]:
%%time
s_list = np.arange(34., 46.1, 2.)
p_list = []; d_list = []; v_list = []
for s in s_list:
    eur_call.update(initial_value=s)
    p_list.append(eur_call.present_value(fixed_seed=True))
    d_list.append(eur_call.delta())
    v_list.append(eur_call.vega())

In [ ]:
from plot_option_stats import plot_option_stats

In [ ]:
plot_option_stats(s_list, p_list, d_list, v_list)

In [ ]:
payoff_func = 'np.maximum(0.33 * (maturity_value + max_value) - 40, 0)'

In [ ]:
eur_as_call = valuation_mcs_european('eur_as_call', underlying=gbm,
                            mar_env=me_call, payoff_func=payoff_func)

In [ ]:
%%time
s_list = np.arange(34., 46.1, 2.)
p_list = []; d_list = []; v_list = []
for s in s_list:
    eur_as_call.update(s)
    p_list.append(eur_as_call.present_value(fixed_seed=True))
    d_list.append(eur_as_call.delta())
    v_list.append(eur_as_call.vega())

In [ ]:
plot_option_stats(s_list, p_list, d_list, v_list)

### American Options

In [ ]:
me_gbm = market_environment('me_gbm', dt.datetime(2020, 1, 1))

In [ ]:
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('volatility', 0.2)
me_gbm.add_constant('final_date', dt.datetime(2021, 12, 31))
me_gbm.add_constant('currency', 'EUR')
me_gbm.add_constant('frequency', 'W')
me_gbm.add_constant('paths', 50000)

In [ ]:
csr = constant_short_rate('csr', 0.06)

In [ ]:
me_gbm.add_curve('discount_curve', csr)

In [ ]:
gbm = geometric_brownian_motion('gbm', me_gbm)

In [ ]:
payoff_func = 'np.maximum(strike - instrument_values, 0)'

In [ ]:
me_am_put = market_environment('me_am_put', dt.datetime(2020, 1, 1))

In [ ]:
me_am_put.add_constant('maturity', dt.datetime(2020, 12, 31))
me_am_put.add_constant('strike', 40.)
me_am_put.add_constant('currency', 'EUR')

In [ ]:
from valuation_mcs_american import valuation_mcs_american

In [ ]:
am_put = valuation_mcs_american('am_put', underlying=gbm,
                    mar_env=me_am_put, payoff_func=payoff_func)

In [ ]:
%time am_put.present_value(fixed_seed=True, bf=5)

In [ ]:
%%time
ls_table = []
for initial_value in (36., 38., 40., 42., 44.):
    for volatility in (0.2, 0.4):
        for maturity in (dt.datetime(2020, 12, 31),
                         dt.datetime(2021, 12, 31)):
            am_put.update(initial_value=initial_value,
                          volatility=volatility,
                          maturity=maturity)
            ls_table.append([initial_value,
                             volatility,
                             maturity,
                             am_put.present_value(bf=5)])

In [ ]:
print('S0  | Vola | T | Value')
print(22 * '-')
for r in ls_table:
    print('%d  | %3.1f  | %d | %5.3f' %
          (r[0], r[1], r[2].year - 2019, r[3]))

In [ ]:
am_put.update(initial_value=36.)
am_put.delta()

In [ ]:
am_put.vega()

## Portfolios

### Position

In [ ]:
from dx_valuation import *

In [ ]:
me_gbm = market_environment('me_gbm', dt.datetime(2020, 1, 1))

In [ ]:
me_gbm.add_constant('initial_value', 36.)
me_gbm.add_constant('volatility', 0.2)
me_gbm.add_constant('currency', 'EUR')

In [ ]:
me_gbm.add_constant('model', 'gbm')

In [ ]:
from derivatives_position import derivatives_position

In [ ]:
me_am_put = market_environment('me_am_put', dt.datetime(2020, 1, 1))

In [ ]:
me_am_put.add_constant('maturity', dt.datetime(2020, 12, 31))
me_am_put.add_constant('strike', 40.)
me_am_put.add_constant('currency', 'EUR')

In [ ]:
payoff_func = 'np.maximum(strike - instrument_values, 0)'

In [ ]:
am_put_pos = derivatives_position(
             name='am_put_pos',
             quantity=3,
             underlying='gbm',
             mar_env=me_am_put,
             otype='American',
             payoff_func=payoff_func)

In [ ]:
am_put_pos.get_info()

### Portfolio

In [ ]:
me_jd = market_environment('me_jd', me_gbm.pricing_date)

In [ ]:
# add jump diffusion specific parameters
me_jd.add_constant('lambda', 0.3)
me_jd.add_constant('mu', -0.75)
me_jd.add_constant('delta', 0.1)
# add other parameters from gbm
me_jd.add_environment(me_gbm)

In [ ]:
# needed for portfolio valuation
me_jd.add_constant('model', 'jd')

In [ ]:
me_eur_call = market_environment('me_eur_call', me_jd.pricing_date)

In [ ]:
me_eur_call.add_constant('maturity', dt.datetime(2020, 6, 30))
me_eur_call.add_constant('strike', 38.)
me_eur_call.add_constant('currency', 'EUR')

In [ ]:
payoff_func = 'np.maximum(maturity_value - strike, 0)'

In [ ]:
eur_call_pos = derivatives_position(
             name='eur_call_pos',
             quantity=5,
             underlying='jd',
             mar_env=me_eur_call,
             otype='European',
             payoff_func=payoff_func)

In [ ]:
underlyings = {'gbm': me_gbm, 'jd' : me_jd}
positions = {'am_put_pos' : am_put_pos,
             'eur_call_pos' : eur_call_pos}

In [ ]:
csr = constant_short_rate('csr', 0.06)

In [ ]:
val_env = market_environment('general', me_gbm.pricing_date)
val_env.add_constant('frequency', 'W')
val_env.add_constant('paths', 25000)
val_env.add_constant('starting_date', val_env.pricing_date)
val_env.add_constant('final_date', val_env.pricing_date)
val_env.add_curve('discount_curve', csr)

In [ ]:
from derivatives_portfolio import derivatives_portfolio

In [ ]:
portfolio = derivatives_portfolio(
                name='portfolio',
                positions=positions,
                val_env=val_env,
                assets=underlyings,
                fixed_seed=False)

In [ ]:
%time portfolio.get_statistics(fixed_seed=False)

In [ ]:
portfolio.get_statistics(fixed_seed=False)[
    ['pos_value', 'pos_delta', 'pos_vega']].sum()

In [ ]:
# portfolio.get_positions()

In [ ]:
portfolio.valuation_objects['am_put_pos'].present_value()

In [ ]:
portfolio.valuation_objects['eur_call_pos'].delta()

In [ ]:
path_no = 888
path_gbm = portfolio.underlying_objects[
    'gbm'].get_instrument_values()[:, path_no]
path_jd = portfolio.underlying_objects[
    'jd'].get_instrument_values()[:, path_no]

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(portfolio.time_grid, path_gbm, 'r', label='gbm')
plt.plot(portfolio.time_grid, path_jd, 'b', label='jd')
plt.xticks(rotation=30)
plt.legend(loc=0)

In [ ]:
correlations = [['gbm', 'jd', 0.9]]

In [ ]:
port_corr = derivatives_portfolio(
                name='portfolio',
                positions=positions,
                val_env=val_env,
                assets=underlyings,
                correlations=correlations,
                fixed_seed=True)

In [ ]:
port_corr.get_statistics()

In [ ]:
path_gbm = port_corr.underlying_objects['gbm'].\
            get_instrument_values()[:, path_no]
path_jd = port_corr.underlying_objects['jd'].\
            get_instrument_values()[:, path_no]

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(portfolio.time_grid, path_gbm, 'r', label='gbm')
plt.plot(portfolio.time_grid, path_jd, 'b', label='jd')
plt.xticks(rotation=30)
plt.legend(loc=0);

In [ ]:
pv1 = 5 * port_corr.valuation_objects['eur_call_pos'].\
            present_value(full=True)[1]
pv1

In [ ]:
pv2 = 3 * port_corr.valuation_objects['am_put_pos'].\
            present_value(full=True)[1]
pv2

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist([pv1, pv2], bins=25,
         label=['European call', 'American put']);
plt.axvline(pv1.mean(), color='r', ls='dashed',
            lw=1.5, label='call mean = %4.2f' % pv1.mean())
plt.axvline(pv2.mean(), color='r', ls='dotted',
            lw=1.5, label='put mean = %4.2f' % pv2.mean())
plt.xlim(0, 80); plt.ylim(0, 10000)
plt.legend();

In [ ]:
pvs = pv1 + pv2
plt.figure(figsize=(10, 6))
plt.hist(pvs, bins=50, label='portfolio');
plt.axvline(pvs.mean(), color='r', ls='dashed',
            lw=1.5, label='mean = %4.2f' % pvs.mean())
plt.xlim(0, 80); plt.ylim(0, 7000)
plt.legend();

In [ ]:
pvs.std()

In [ ]:
pv1 = (5 * portfolio.valuation_objects['eur_call_pos'].
            present_value(full=True)[1])
pv2 = (3 * portfolio.valuation_objects['am_put_pos'].
            present_value(full=True)[1])
(pv1 + pv2).std()

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>
<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>